### Understanding Context Engineering

Context Engineering is new term gaining traction in the AI world. The conversation is shifting from "prompt engineering" to a broader, more powerful concept: Context Engineering. Tobi Lutke describes it as "the art of providing all the context for the task to be plausibly solvable by the LLM.” and he is right.

We are seeing that the main thing that determines whether an Agents succeeds or fails is the quality of the context you give it. Most agent failures are not model failures anyemore, they are context failures.

```Context Engineering is the discipline of designing and building dynamic systems that provides the right information and tools, in the right format, at the right time, to give a LLM everything it needs to accomplish a task.```

give this a read: https://www.philschmid.de/context-engineering

### Let's Build Digital Twin

#### Step 0 - Pushover(send notification to mobile or phone)

In [1]:
from dotenv import load_dotenv
from pypdf import PdfReader
from openai import OpenAI
import os
import json
import requests
import gradio as gr

In [2]:
load_dotenv(override = True)
openai = OpenAI()

In [15]:
# for Pushover
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

#  IK the normal way is easier to read, I wanted to learn a different way
print("Pushover user found and looks good" if pushover_user.startswith("u") else "Check USER again") if pushover_user else "Pushover user not found"
print("Pushover token found and looks good" if pushover_token.startswith("a") else "Check TOKEN again") if pushover_token else "Pushover token not found"   

Pushover user found and looks good
Pushover token found and looks good


In [16]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data = payload)

In [17]:
push("DON'T TALKK MEEE")

Push: DON'T TALKK MEEE


In [18]:
def record_user_details(email, name="(Name not provided)", notes="(Notes not provided)"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [28]:
def record_unknown_questions(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [20]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record a user is interested in being in touch and provide an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided"},
            "notes": {"type": "string", "description": "Additional info about the conversation that's work recording to give context"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [21]:
record_unknown_questions_json = {
    "name": "record_unknown_questions",
    "description": "Use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question" : {"type": "string", "description": "The question that couldn't be answered"}
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [22]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_questions_json}
]

In [39]:
def handle_tool_calls_with_manual_if (tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(tool_name, arguments)
        # print(f"Tool called: {tool_name}", flush=True)

        # if tool_name == "record_user_details":
        #         result = record_user_details(**arguments)
        # elif tool_name == "record_unknown_question":
        #     result = record_unknown_questions(**arguments)

        # results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    # return results 

#### Using Python built-in globals()
Python has a dictionary that gives us access to all global functions.

Sidenote: for sure when we deploy, we will use this in a more protected way..

In [40]:
globals()["record_unknown_questions"]("this is a really hard question")

Push: Recording this is a really hard question asked that I couldn't answer


'OK'

In [ ]:
# This gives us a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name) # Not secure way to let llm give access to all functions
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [38]:
reader = PdfReader("twin/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [41]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


In [43]:
def chat(message, history):
    messages = {"role": "system", "content": system_prompt} + history + {"role": "user", "content": message}
    response = openai.chat.completions.create(
        model = "gpt-5.4-mini", messages = messages
    )
    while response.choices[0].finish_reason == "tool_calls":
        tool_calls = response.choices[0].message
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(
            model = "gpt-5.4-mini", messages = messages, tools = tools
        )
    return response.choices[0].message.content

In [44]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Turning into Python modules

I've turned the code in the lab into python modules; that's a great practice to do after you've completed experiments in the Notebook.

You could put all the code above into 1 python script. But it's nicer to organize the code into different modules for different concerns, and that's what I've done:

`context.py` loads in the static data and constructs the System Prompt

`tools.py` contains all the code to manage and call tools, with their associated json

`app.py` contains the Gradio app and OpenAI call.

`styles.py` contains styles to apply to Gradio and this was entirely written by Claude Code!

You could have a stab at doing this yourself, then compare with my versions.

Then to try it out, open a terminal in Cursor:

`cd 1_foundations`  
`cd twin`  
`uv run app.py`